# LegalQA Task 2 — Canonical Reproducible Dual-T4 Kaggle Pipeline (Stack A / API 16)
Production LegalQA training, validation, and inference pipeline on Kaggle Dual NVIDIA T4 GPUs.
- **Stack A Production Architecture**: Exact/Similar QA Memory -> BM25S (mmap) + Dense DEk21 v2 (FP16 GPU top-K) -> RRF Fusion -> Task-Tuned BGE Reranker -> Structured Evidence Packer -> Qwen2.5-3B-Instruct (4-bit QLoRA with Liger fused-linear CE) -> Candidate Ensemble & Selector Guardrail.
- **Hardware Layout**: Dual NVIDIA T4 (GPU 0: Qwen Generator | GPU 1: DEk21 Dense + BGE Reranker | CPU: BM25S + QA Memory + Selector).

In [ ]:
# Cell 1 — Execution Profile & Authoritative Production Selection Configuration
import os, sys

# Transformers 5.0.0 uses async tensor materialization by default.
# On T4 + on-the-fly bitsandbytes 4-bit loading this can transiently
# materialize weights onto GPU faster than quantization consumes them,
# causing an OOM before QLoRA training starts.
os.environ["HF_DEACTIVATE_ASYNC_LOAD"] = "1"
print("Transformers async model loading: DISABLED for T4-safe QLoRA load")

SEED = 42
ALLOW_SINGLE_GPU_SMOKE = False  # Strict dual-T4 by default; set True only for explicit single-GPU smoke testing
ALLOW_UNVALIDATED_FINAL = False  # Fail-safe: final profile refuses UNVALIDATED config unless explicitly overridden

# Execution Profiles (V16):
#   "generator_probe_worstcase": 3 optimizer steps on worst-case token lengths, validates VRAM
#   "generator_probe_endurance": 30 optimizer steps on full pool, validates memory stability
#   "screen_fold0": Excludes fold 0 from training, runs S0/S1/S2 bake-off, and generates promotion_report.json
#   "final_train_and_submit": Trains only promoted components on ALL allowed data and creates public submission
#   "reuse_final_checkpoints_and_submit": Reuses verified final all-data checkpoints and creates public submission
EXECUTION_PROFILE = "generator_probe_worstcase"  # V16 authoritative T4 generator probe profile

# Staging path configuration
PRODUCTION_CONFIG_PATH = "configs/production_selection.yaml"

print("===========================================================")
print(f"COMMITTED EXECUTION PROFILE: {EXECUTION_PROFILE}")
print(f"RANDOM SEED:                 {SEED}")
print(f"PRODUCTION CONFIG PATH:      {PRODUCTION_CONFIG_PATH}")
print("===========================================================")

In [ ]:
# Cell 2 — Environment, Secrets & Hardware Verification
import os, sys, gc, glob, json, zipfile, re, math, time, subprocess
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

# Set deterministic seeds
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Securely retrieve HuggingFace Token from Kaggle Secrets
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
    if HF_TOKEN:
        os.environ["HF_TOKEN"] = HF_TOKEN
        os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
        print("HF_TOKEN securely loaded from Kaggle Secrets.")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
    if HF_TOKEN:
        print("HF_TOKEN detected from environment.")
    else:
        print("Notice: HF_TOKEN secret not present; using public weights.")

# Dual-GPU Device Allocation & Preflight Gate (P0-3)
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for Kaggle GPU execution but torch.cuda.is_available() is False.")

gpu_count = torch.cuda.device_count()
print(f"CUDA GPUs Detected: {gpu_count}")

if gpu_count < 2 and not ALLOW_SINGLE_GPU_SMOKE:
    raise RuntimeError(
        f"Production execution requires >=2 CUDA GPUs (Dual-T4), but found {gpu_count}. "
        f"Set ALLOW_SINGLE_GPU_SMOKE=True only for explicit single-GPU smoke testing."
    )

if gpu_count >= 2:
    GEN_DEVICE = "cuda:0"
    RETRIEVAL_DEVICE = "cuda:1"
elif gpu_count == 1:
    GEN_DEVICE = "cuda:0"
    RETRIEVAL_DEVICE = "cuda:0"
else:
    GEN_DEVICE = "cpu"
    RETRIEVAL_DEVICE = "cpu"

for i in range(gpu_count):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name} | VRAM: {p.total_memory / (1024**3):.1f} GB | Compute: sm_{p.major}{p.minor}")
print(f"Hardware Allocation -> Generator: {GEN_DEVICE} | Retrieval/Reranker: {RETRIEVAL_DEVICE}")

In [ ]:
# Cell 3 — Strict Packaged Code & Runtime Manifest Verification (V16 Release Binding)
from pathlib import Path
import os, sys, glob, json

# 1. Notebook-Owned Runtime Release Binding Contract (V16)
REQUIRED_RUNTIME_API_VERSION = 16
EXPECTED_CODE_GIT_SHA = None  # Set to specific commit SHA for strict release runs if desired

# 2. Deterministic Strict Code Root Bootstrap (No development fallback)
_bootstrap_roots = sorted({
    os.path.abspath(p)
    for p in glob.glob("/kaggle/input/**/code/LegalQA", recursive=True)
    if os.path.isdir(os.path.join(p, "src"))
    and os.path.isdir(os.path.join(p, "scripts"))
})

if len(_bootstrap_roots) != 1:
    raise RuntimeError(
        f"Expected exactly one packaged LegalQA code root under /kaggle/input, found {_bootstrap_roots}"
    )

sys.path.insert(0, _bootstrap_roots[0])

from src.task2.runtime_integrity import (
    resolve_packaged_code_root,
    validate_runtime_manifests,
)
from src.task2.path_resolver import find_runtime_roots, find_qwen_model_dir, resolve_runtime_paths
from src.task2.production_config import load_production_selection
from src.task2.pipeline.profiles import resolve_execution_profile

resolved_code_root = resolve_packaged_code_root("/kaggle/input", strict=True)
runtime_root_from_code = str(Path(resolved_code_root).resolve().parents[1])
print(f"Packaged Code Root resolved: {resolved_code_root}")
print(f"Runtime Root from Code:     {runtime_root_from_code}")

# 3. Unconditional Manifest Provenance Verification (Validated BEFORE resolve_runtime_paths)
RUNTIME_PROVENANCE = validate_runtime_manifests(
    runtime_root=runtime_root_from_code,
    code_root=resolved_code_root,
    expected_api_version=REQUIRED_RUNTIME_API_VERSION,
    expected_git_sha=EXPECTED_CODE_GIT_SHA,
)

# 4. Strict Deterministic Runtime Paths (Unconditionally strict with allow_remote_model_download=False)
paths = resolve_runtime_paths(
    "/kaggle/input",
    strict=True,
    allow_remote_model_download=False,
)

# 5. Cross-Check Runtime Roots
assert os.path.realpath(paths["runtime_root"]) == os.path.realpath(runtime_root_from_code), (
    f"Runtime root mismatch: resolved '{paths['runtime_root']}' vs code parent '{runtime_root_from_code}'"
)

# 6. Resolve production configuration and execution profile
if os.path.exists(PRODUCTION_CONFIG_PATH):
    resolved_prod_cfg_path = PRODUCTION_CONFIG_PATH
else:
    resolved_prod_cfg_path = os.path.join(resolved_code_root, "configs", "production_selection.yaml")

PRODUCTION_CFG = load_production_selection(resolved_prod_cfg_path)
ACTIVE_PROFILE = resolve_execution_profile(
    EXECUTION_PROFILE,
    production_cfg=PRODUCTION_CFG,
    allow_unvalidated_final=ALLOW_UNVALIDATED_FINAL,
)

print(f"Resolved Active Profile: {ACTIVE_PROFILE.name}")
print(f" - Train Reranker:      {ACTIVE_PROFILE.run_reranker_training}")
print(f" - Train Generator:     {ACTIVE_PROFILE.run_generator_training}")
print(f" - Probe Selection:     {ACTIVE_PROFILE.probe_selection}")
print(f" - Max Generator Steps: {ACTIVE_PROFILE.max_generator_steps}")

In [ ]:
# Cell 4 — Dependency Compatibility & Environment Bootstrap (V16)
import scripts.bootstrap_kaggle_env as bstrap

bstrap.print_preinstalled_environment()
BOOTSTRAP_RESULT = bstrap.bootstrap_dependencies()
bstrap.verify_runtime_imports(strict=True)
bstrap.save_bootstrap_manifest(
    bootstrap_result=BOOTSTRAP_RESULT,
)

In [ ]:
# Cell 5 — Execute Pipeline Runner (Decoupled Stage Orchestration)
from src.task2.pipeline.runner import run_pipeline

paths["public_test_path"] = os.path.join(paths["data_dir"], "public-official.json")
if not os.path.exists(paths["public_test_path"]):
    paths["public_test_path"] = os.path.join(paths["runtime_root"], "public-official.json")
if not os.path.exists(paths["public_test_path"]):
    paths["public_test_path"] = "artifacts/raw/public-official.json"

PIPELINE_OUTPUTS = run_pipeline(
    profile=ACTIVE_PROFILE,
    paths=paths,
    production_cfg=PRODUCTION_CFG,
    gen_device=GEN_DEVICE,
    retrieval_device=RETRIEVAL_DEVICE,
    output_dir="/kaggle/working",
    seed=SEED,
    code_root=resolved_code_root,
    allow_single_gpu=ALLOW_SINGLE_GPU_SMOKE,
)

print("\nPipeline execution completed successfully.")
print("Stages executed:", list(PIPELINE_OUTPUTS.get("stages", {}).keys()))

In [ ]:
# Cell 6 — Export & Handoff Summary
import glob

print("=== Run Directory Artifacts ===")
for p in sorted(glob.glob("/kaggle/working/**", recursive=True)):
    if os.path.isfile(p):
        print(f" - {p} ({os.path.getsize(p)/1024:.1f} KB)")

print(f"\nExecution profile '{EXECUTION_PROFILE}' finished cleanly.")